In [10]:
import re
import pandas as pd

In [11]:
file_path = 'data/NCT03860142.ann'

In [12]:
df = pd.read_csv(file_path, sep='^([^\s]*)\s', engine='python', header=None).drop(0, axis=1)

# Umbenennen der Spalten
df.columns = ['ID', 'Details']

# Extrahieren der Zeilen, die mit 'T' oder 'E' beginnen
t_e_df = df[df['ID'].str.startswith(('T', 'E'))]

# Dictionary zur Speicherung der Textinhalte von T- und E-Entitäten
text_dict = {}

# Textinhalte extrahieren und in das Dictionary einfügen
for index, row in t_e_df.iterrows():
    entity_id = row['ID']
    details = row['Details']
    text_content = details.split('\t')[-1]
    text_dict[entity_id] = text_content

# Funktion, um den vollständigen Text einer Entität zu erhalten
def get_full_text(entity_id):
    if entity_id in text_dict:
        text_content = text_dict[entity_id]
        # Wenn die Entität 'E' ist, folge der 'T' Referenz innerhalb
        if entity_id.startswith('E'):
            sub_entity_id = re.search(r'\b(T\d+)\b', text_content)
            if sub_entity_id:
                return get_full_text(sub_entity_id.group(1))
        return text_content
    return entity_id

# Extrahieren der Zeilen, die Beziehungen darstellen
rel_df = df[df['ID'].str.startswith('R')]

# Regular expression to match relationships
rel_pattern = re.compile(r'^(R\d+)\t(And|Or) Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')

# Beziehungen finden und speichern
relationships = []
for index, row in rel_df.iterrows():
    line = f"{row['ID']}\t{row['Details']}"
    rel_match = rel_pattern.match(line)
    if rel_match:
        rel_id, rel_type, arg1, arg2 = rel_match.groups()
        relationships.append((rel_type, arg1, arg2))

# Convert IDs to actual texts recursively
rel_texts = []
for rel_type, arg1, arg2 in relationships:
    arg1_text = get_full_text(arg1)
    arg2_text = get_full_text(arg2)
    rel_texts.append((rel_type, arg1_text, arg2_text))

# Ausgabe der Beziehungen mit den Texten der Entitäten
for rel_type, arg1_text, arg2_text in rel_texts:
    print(f"{rel_type} between '{arg1_text}' and '{arg2_text}'")

And between 'Children' and 'age'
And between '≥ 24 months' and '≤ 36 months'
Or between 'Full-term' and 'prematurely'
And between 'Children' and 'congenital pathologies'
Or between 'neurological' and 'developmental'
And between 'Children' and 'pathologies'
And between 'Children' and 'ENT deformities'


In [11]:
from brat_parser import get_entities_relations_attributes_groups
entities, relations, attributes, groups = get_entities_relations_attributes_groups(file_path)

In [23]:
attributes    

{'A1': Attribute(id='A1', type='Life-Stage-And-Gender-Type', target='T3', values=('child',)),
 'A2': Attribute(id='A2', type='Life-Stage-And-Gender-Type', target='T4', values=('child',)),
 'A3': Attribute(id='A3', type='Life-Stage-And-Gender-Type', target='T5', values=('child',)),
 'A4': Attribute(id='A4', type='Life-Stage-And-Gender-Type', target='T6', values=('child',)),
 'A5': Attribute(id='A5', type='Eq-Operator-Value', target='T9', values=('GTEQ',)),
 'A6': Attribute(id='A6', type='Eq-Operator-Value', target='T10', values=('LTEQ',)),
 'A7': Attribute(id='A7', type='Eq-Operator-Value', target='T11', values=('LT',)),
 'A8': Attribute(id='A8', type='Eq-Temporal-Unit-Value', target='T12', values=('month',)),
 'A9': Attribute(id='A9', type='Eq-Temporal-Unit-Value', target='T13', values=('month',)),
 'A10': Attribute(id='A10', type='Location-Value', target='T27', values=('hospital',)),
 'A11': Attribute(id='A11', type='Family-Member-Type', target='E26', values=('parent',))}

In [22]:
relations

{'R1': Relation(id='R1', type='Equivalent-To', subj='E12', obj='E4'),
 'R2': Relation(id='R2', type='Equivalent-To', subj='E14', obj='E13'),
 'R3': Relation(id='R3', type='And', subj='T3', obj='E8'),
 'R4': Relation(id='R4', type='And', subj='E9', obj='E10'),
 'R5': Relation(id='R5', type='Or', subj='E6', obj='E5'),
 'R6': Relation(id='R6', type='And', subj='T4', obj='E13'),
 'R7': Relation(id='R7', type='Or', subj='E17', obj='E18'),
 'R8': Relation(id='R8', type='And', subj='T5', obj='E1'),
 'R9': Relation(id='R9', type='And', subj='T6', obj='E23')}

In [20]:
def get_full_text(entity_id):
    if entity_id in entities:
        return entities[entity_id].text
    return entity_id

# Neues Dictionary zur Speicherung der Beziehungen mit Texten
relationship_texts = {}

# Durch alle Beziehungen iterieren
for rel_id, rel in relations.items():
    if rel.type in ['And', 'Or']:
        subj_text = get_full_text(rel.subj)
        obj_text = get_full_text(rel.obj)
        if rel.type not in relationship_texts:
            relationship_texts[rel.type] = []
        relationship_texts[rel.type].append((subj_text, obj_text))

# Ausgabe des neuen Dictionaries
for rel_type, pairs in relationship_texts.items():
    print(f"{rel_type}:")
    for subj_text, obj_text in pairs:
        print(f"  between '{subj_text}' and '{obj_text}'")

# Dictionary zurückgeben
print(relationship_texts)

And:
  between 'Children' and 'E8'
  between 'E9' and 'E10'
  between 'Children' and 'E13'
  between 'Children' and 'E1'
  between 'Children' and 'E23'
Or:
  between 'E6' and 'E5'
  between 'E17' and 'E18'
{'And': [('Children', 'E8'), ('E9', 'E10'), ('Children', 'E13'), ('Children', 'E1'), ('Children', 'E23')], 'Or': [('E6', 'E5'), ('E17', 'E18')]}
